# Predicting Smartphone Addiction — OOF Stacking Project

Этот проект решает задачу бинарного ранжирования пользователей по вероятности принадлежности к классу `addicted_label` на данных Kaggle Playground Series S6E8.

Значение задачи на практике — построение метрики риска, которая позволяет упорядочить пользователей по степени серьёзности проблемы неправильного использования телефона. В реальном продукте такая метрика могла бы использоваться для разумного подхода к A/B-тестированию, созданию персональных рекомендаций или дополнительному анализу. В рамках соревнования задача оценивается по ROC-AUC, поэтому основное внимание уделяется качеству ранжирования, а не выбору фиксированного порога классификации.

Итог проекта — **216 место из 3520 участников/решений рейтинга (Top ~6%)**.

**У моего решения два уровня:**

- На первом уровне используются публично доступные OOF/test-предсказания разных моделей и ансамблей. 
- Вторая часть решения сосредоточена на построении воспроизводимого stacking-пайплайна: проверке и объединении OOF-источников, фильтрации почти дублирующихся сигналов, rank-based преобразованиях, честной OOF-валидации метамодели и сборке финального прогноза.

## 1. Импорты и конфигурация

Для воспроизводимости результата зафиксированы:

- размер и порядок строк competition данных;
- схема `StratifiedKFold`;
- порядок загруженных OOF-моделей;
- критерий удаления почти одинаковых predictions;
- Rank-Gauss преобразование;
- параметры Logistic Regression;
- финальные веса competition blend.

По умолчанию notebook рассчитан на Kaggle-окружение с данными в `/kaggle/input`. Для локального запуска корневую директорию входных данных можно переопределить через переменную окружения `S6E8_INPUT`.

In [1]:
import glob
import os

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


N_TRAIN = 691_369
N_TEST = 296_302
INPUT = os.environ.get("S6E8_INPUT", "/kaggle/input")


def find_file(filename, must_contain=None):
    """Ищет файл по всему Kaggle input.

    must_contain позволяет выбрать нужный dataset, если файлов
    с одинаковым именем несколько.
    """
    hits = glob.glob(f"{INPUT}/**/{filename}", recursive=True)

    if must_contain:
        hits = [
            path
            for path in hits
            if must_contain.lower() in path.lower()
        ]

    if not hits:
        raise FileNotFoundError(
            f"{filename!r} (containing {must_contain!r}) "
            f"not found under {INPUT}"
        )

    return sorted(hits, key=len)[0]


def find_dir(folder):
    """Ищет директорию подключённого Kaggle dataset."""
    hits = [
        path
        for path in glob.glob(f"{INPUT}/**/{folder}", recursive=True)
        if os.path.isdir(path)
    ]

    return sorted(hits, key=len)[0] if hits else None


## 2. Данные и схема кросс-валидации

Целевая переменная `addicted_label` - задача бинарной классификации с целевой метрикой ROC-AUC 
Поэтому модель должна прежде всего правильно **ранжировать** объекты: пользователи с положительным классом должны в среднем получать более высокий score, чем пользователи с отрицательным.

Для построения OOF-прогнозов используется фиксированная стратифицированная схема:

```python
StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```

Для этого проекта особенно критичен порядок строк. Большинство внешних OOF-файлов представлено массивами `.npy` без `id`, поэтому `oof[i]` должен соответствовать той же строке из `train`, для которой предсказание было создано исходной моделью. Любая сортировка или перестановка train до объединения OOF нарушит порядок строк и сделает ансамбль бесполезным.

In [2]:
train = pd.read_csv(
    find_file("train.csv", "playground-series-s6e8")
)
test = pd.read_csv(
    find_file("test.csv", "playground-series-s6e8")
)

assert len(train) == N_TRAIN
assert len(test) == N_TEST

y = train["addicted_label"].to_numpy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

folds = list(
    cv.split(np.zeros(N_TRAIN), y)
)

print(
    f"{len(train):,} train rows, "
    f"{len(test):,} test rows, "
    f"positive rate {y.mean():.4f}"
)


691,369 train rows, 296,302 test rows, positive rate 0.7094


## 3. Собираем библиотеку OOF/test-предсказаний

Вместо исходных признаков метамодель получает предсказания базовых моделей как совершенно новые признаки.

Для каждой модели нужны два массива:

- **OOF predictions** для train предсказания каждой строки получено моделью, которая не обучалась на целевой переменной этой строки;
- **test predictions** — предсказания той же модели для соревновательного test.

Получившаяся матрица признаков уже описывает не исходное поведение пользователя, а то, как разные модели интерпретируют каждый объект. Публичные Kaggle-артефакты используют несколько вариантов именования файлов, поэтому loader приводит их к единому представлению и одновременно проверяет длину массивов и отсутствие `NaN/inf`.

In [3]:
def load_pairs(
    root,
    prefix,
    test_prefixes=("test_", "testpred_", "tep_"),
):
    """Загружает пары oof_<name>.npy + test_<name>.npy."""
    result = {}

    pattern = os.path.join(root, "**", "oof_*.npy")

    for oof_path in glob.glob(pattern, recursive=True):
        name = os.path.basename(oof_path)[4:-4]

        candidates = [
            os.path.join(
                os.path.dirname(oof_path),
                test_prefix + name + ".npy",
            )
            for test_prefix in test_prefixes
        ]

        test_path = next(
            (
                candidate
                for candidate in candidates
                if os.path.exists(candidate)
            ),
            None,
        )

        if test_path is None:
            continue

        oof = np.load(oof_path).astype(np.float64)
        test_pred = np.load(test_path).astype(np.float64)

        valid_shapes = (
            oof.shape == (N_TRAIN,)
            and test_pred.shape == (N_TEST,)
        )
        finite_values = (
            np.isfinite(oof).all()
            and np.isfinite(test_pred).all()
        )

        if valid_shapes and finite_values:
            result[prefix + name] = (oof, test_pred)

    return result


SOURCES = [
    ("s6e8-oof-library-47-models", "sz_"),
    ("s6e8-oof-library-11-members", "nn_"),
    ("s6e8-mask-augmented-oof-library", "ma_"),
    ("s6e8-full-best-blend-npy", "tam_"),
    ("s6e8-adarsh-oof-library", "a_"),
    ("s6e8-golem-oof-library", "golem_"),
    ("s6e8-fm-lattice-blend-members", "fm_"),
    ("s6e8-150-fusion-local-members", "hb_"),
    ("s6e8-catstrall-member", "x_"),
    ("s6e8-catstr-aug16", "mk_"),
]

members = {}

for folder, prefix in SOURCES:
    root = find_dir(folder)
    loaded = load_pairs(root, prefix) if root else {}

    members.update(loaded)

    suffix = "" if root else "   NOT ATTACHED"
    print(f"{folder:36s} {len(loaded):3d}{suffix}")


s6e8-oof-library-47-models            74
s6e8-oof-library-11-members           11
s6e8-mask-augmented-oof-library        9
s6e8-full-best-blend-npy               9
s6e8-adarsh-oof-library               22
s6e8-golem-oof-library                 7
s6e8-fm-lattice-blend-members          5
s6e8-150-fusion-local-members         17
s6e8-catstrall-member                  5
s6e8-catstr-aug16                      1


### Дополнительные форматы опубликованных предсказаний

Не все опубликованные OOF-библиотеки хранят предсказания одинаково. Основная часть представлена парами `.npy`, однако некоторые источники содержат:

- parquet-таблицы, где один столбец соответствует одной модели;
- двумерные `.npy`-матрицы вида `(n_rows, n_models)`.

На этом этапе разные форматы приводятся к одной структуре `members`, после чего формируются две согласованные матрицы:

- `OOF` — признаки обучающей выборки для метамодели;
- `TST` — соответствующие признаки для метамодели признаки соревновательного test.

Для каждого OOF-столбца дополнительно рассчитывается индивидуальный ROC-AUC. Его значение используется не только для проверки качества, но и далее — при выборе между почти идентичными моделями.

In [4]:
# Библиотека в parquet: один столбец = одна модель
bolt = find_dir("s6e8-oof-prediction-library") or ""

if bolt and os.path.exists(f"{bolt}/oof_predictions.parquet"):
    oof_df = pd.read_parquet(
        f"{bolt}/oof_predictions.parquet"
    )
    test_df = pd.read_parquet(
        f"{bolt}/test_predictions.parquet"
    )

    for column in oof_df.columns:
        if column != "id" and column in test_df:
            members[f"bolt_{column}"] = (
                oof_df[column].to_numpy(float),
                test_df[column].to_numpy(float),
            )


# 50 слабых моделей хранятся одной двумерной матрицей
weak = find_dir("s6e8-50-weakest-oof-models") or ""

if weak and os.path.exists(f"{weak}/oof.npy"):
    weak_oof = np.load(
        f"{weak}/oof.npy",
        mmap_mode="r",
    )
    weak_test = np.load(
        f"{weak}/test.npy",
        mmap_mode="r",
    )

    for j in range(weak_oof.shape[1]):
        members[f"weak_{j:02d}"] = (
            np.asarray(weak_oof[:, j], float),
            np.asarray(weak_test[:, j], float),
        )


# Сортировка важна для воспроизводимости порядка столбцов.
names = sorted(members)

OOF = np.column_stack(
    [members[name][0] for name in names]
)
TST = np.column_stack(
    [members[name][1] for name in names]
)

member_auc = pd.Series(
    [
        roc_auc_score(y, OOF[:, j])
        for j in range(OOF.shape[1])
    ],
    index=names,
)

print(
    f"{len(names)} members, "
    f"OOF AUC from {member_auc.min():.5f} "
    f"to {member_auc.max():.5f}"
)


207 members, OOF AUC from 0.91880 to 0.96987


## 4. Удаляем почти одинаковые модели

Большая OOF-библиотека полезна только тогда, когда её модели приносят разные сигналы. Две модели могут иметь разные названия и даже разные алгоритмы, но практически одинаково ранжировать все объекты.

Для ROC-AUC особенно важен именно порядок предсказаний, поэтому сходство оценивается после преобразования каждого столбца в percentile ranks.

Если rank-correlation двух моделей превышает `0.9995`, они считаются почти дубликатами. Из такой пары сохраняется модель с более высоким OOF ROC-AUC.

Цель этого шага — не максимизировать количество моделей, а оставить **качественный и достаточно разнообразный набор сигналов** для второго уровня.

In [5]:
def pct_rank(values):
    """Percentile rank в интервале (0, 1)."""
    return (
        rankdata(values) - 0.5
    ) / len(values)


R = np.column_stack(
    [
        pct_rank(OOF[:, j])
        for j in range(OOF.shape[1])
    ]
).astype(np.float32)

Rt = np.column_stack(
    [
        pct_rank(TST[:, j])
        for j in range(TST.shape[1])
    ]
).astype(np.float32)


# После стандартизации скалярное произведение даёт
# матрицу корреляций и экономит память.
Z = (
    R - R.mean(axis=0)
) / (
    R.std(axis=0) + 1e-12
)

corr = (Z.T @ Z) / len(Z)
del Z


drop = set()

for i in range(len(names)):
    if names[i] in drop:
        continue

    for j in range(i + 1, len(names)):
        if names[j] in drop:
            continue

        if corr[i, j] <= 0.9995:
            continue

        if member_auc[names[i]] >= member_auc[names[j]]:
            drop.add(names[j])
        else:
            drop.add(names[i])


print(
    f"{len(drop)} near-duplicate members dropped "
    "at rank correlation > 0.9995"
)


8 near-duplicate members dropped at rank correlation > 0.9995


## 5. Убираем self-referential members
Часть публичных предсказаний сама является результатом **blending или stacking** поверх моделей, которые уже присутствуют в OOF-библиотеке.

Если одновременно передать метамодели исходные предсказания и ансамбль, построенный из тех же предсказаний, один и тот же сигнал может быть представлен несколько раз. Это повышает корреляцию признаков и создаёт риск переоценить отдельное семейство моделей.

Поэтому заранее некоторые модели исключаются из пула по имени.

После этого остаётся набор признаков мета-модели, который лучше соответствует основной идее stacking: объединять сильные, но не полностью одинаковые взгляды на одни и те же объекты.

In [6]:
SELF_REFERENTIAL = (
    "naji",
    "sz_naji",
    "v13_anchor",
    "hb_candidate",
)

keep = [
    i
    for i, name in enumerate(names)
    if name not in drop
    and not name.startswith(SELF_REFERENTIAL)
]

names = [names[i] for i in keep]
R = R[:, keep]
Rt = Rt[:, keep]

print(f"{len(names)} members kept")

194 members kept


## 6. Rank-Gauss преобразование

Разные базовые модели могут быть по-разному откалиброваны. Например, две модели способны одинаково хорошо ранжировать пользователей, но одна выдаёт предсказания в диапазоне `0.1–0.9`, а другая — `0.35–0.65`.

Для ROC-AUC абсолютный масштаб таких вероятностей вторичен. Поэтому каждое предсказание сначала преобразуется в percentile rank, то есть в относительную позицию объекта среди остальных.

Далее применяется Rank-Gauss - это сохраняет ранжирование каждой базовой модели, но приводит разные предсказания к сопоставимому масштабу. В таком пространстве линейной метамодели проще учить устойчивые веса между большим количеством коррелирующих предсказаний выбранных моделей.

In [7]:
G = norm.ppf(
    np.clip(R, 1e-7, 1 - 1e-7)
).astype(np.float32)

Gt = norm.ppf(
    np.clip(Rt, 1e-7, 1 - 1e-7)
).astype(np.float32)

print(
    "meta feature matrices:",
    G.shape,
    Gt.shape,
)


meta feature matrices: (691369, 194) (296302, 194)


## 7. Cross-fitted logistic regression

После подготовки OOF-библиотеки задача второго уровня становится существенно проще: нужно определить, каким предсказаниям доверять больше и как их совместно использовать.

В качестве метамодели используется Logistic Regression. Её ограниченная гибкость здесь является преимуществом: базовые модели уже извлекли нелинейные зависимости из исходных данных, а второму уровню в первую очередь требуется устойчиво объединить их сигналы, не переобучаясь на небольших различиях между коррелирующими OOF-столбцами.

Метамодель также оценивается через cross-fitting. На каждом fold:

1. `StandardScaler` обучается только на обучающей выборке meta-features;
2. Logistic Regression обучается на тех же строках;
3. предсказания строятся по validation fold, который модель не видела;
4. после пяти folds формируется полный OOF-вектор второго уровня.

Итоговый OOF ROC-AUC является локальной оценкой качества самого stack. Только после этой оценки метамодель переобучается на всей обучающей выборке и строит предсказания для соревновательного test.

In [8]:
def fit_logistic(
    X_fit,
    y_fit,
    X_pred,
    C=1.0,
):
    scaler = StandardScaler().fit(X_fit)

    model = LogisticRegression(
        C=C,
        max_iter=3000,
        solver="lbfgs",
        tol=1e-5,
    )

    model.fit(
        scaler.transform(X_fit),
        y_fit,
    )

    assert int(np.max(model.n_iter_)) < 3000, (
        "meta-model did not converge"
    )

    return model.predict_proba(
        scaler.transform(X_pred)
    )[:, 1]


oof_meta = np.zeros(N_TRAIN)

for fold_number, (fit_idx, val_idx) in enumerate(folds):
    oof_meta[val_idx] = fit_logistic(
        G[fit_idx],
        y[fit_idx],
        G[val_idx],
    )

    fold_auc = roc_auc_score(
        y[val_idx],
        oof_meta[val_idx],
    )

    print(
        f"fold {fold_number}: "
        f"ROC-AUC = {fold_auc:.6f}"
    )


stack_auc = roc_auc_score(y, oof_meta)
print(f"\nstack OOF AUC = {stack_auc:.6f}")


# После честной OOF-оценки обучаем meta-model
# уже на всех train-строках и прогнозируем Kaggle test.
test_meta = fit_logistic(
    G,
    y,
    Gt,
)


fold 0: ROC-AUC = 0.969514
fold 1: ROC-AUC = 0.970236
fold 2: ROC-AUC = 0.970278
fold 3: ROC-AUC = 0.970805
fold 4: ROC-AUC = 0.969899

stack OOF AUC = 0.970134


## 8. Финальный blend

Предыдущий этап формирует самостоятельный и локально валидируемый stack. Далее начинается отдельный этап для подачи Kaggle-решения.

Для финального соревновательного решения предсказания метамодели смешивается с сильным публичным набором предсказаний. Перед смешиванием предсказания снова переводятся в rank-форму, чтобы веса отражали вклад в итоговое ранжирование, а не различия калибровки. 

Этот этап намеренно отделён от основного ML-пайплайна: прогон модели по тестовой выборке невозможно проверить той же кросс-валидацией, тк для неё нет соответствующих флагов

По этой причине проект сохраняет два результата:
- **base stack** — только локально валидированная метамодель;
- **competition blend** — финальная версия, оптимизированная для Kaggle.

In [9]:
public_1 = pd.read_csv(
    find_file("submission.csv", "vault")
)["addicted_label"].to_numpy()

public_2 = pd.read_csv(
    find_file("submission (1).csv", "vault")
)["addicted_label"].to_numpy()


public = pct_rank(
    (2.9 * public_1 + 0.1 * public_2) / 3.0
)

W = 0.35

final = pct_rank(
    W * pct_rank(test_meta)
    + (1 - W) * public
)


rank_corr = np.corrcoef(
    pct_rank(test_meta),
    public,
)[0, 1]

print(
    "rank correlation, stack vs public submission: "
    f"{rank_corr:.5f}"
)


rank correlation, stack vs public submission: 0.99541


## 9. Сохраняем решения

Ноутбук формирует два submission-файла.

`submission_base_stack.csv` содержит предсказания только cross-fitted stacking-системы. Это наиболее чистое представление локально валидируемой модели проекта.

`submission.csv` содержит финальный набор предсказаний, который дополнительно использует public test prediction.

Разделение этих файлов позволяет не смешивать две разные задачи: оценку качества собственного pipeline метамодели и competition-specific оптимизацию финального решения.

In [ ]:
sub_base = pd.DataFrame(
    {
        "id": test["id"],
        "addicted_label": pct_rank(test_meta),
    }
)

sub_base.to_csv(
    "submission_base_stack.csv",
    index=False,
)


submission = pd.DataFrame(
    {
        "id": test["id"],
        "addicted_label": final,
    }
)

assert len(submission) == N_TEST
assert submission["id"].is_unique

submission.to_csv(
    "submission.csv",
    index=False,
)

print(
    "saved: submission.csv "
    "+ submission_base_stack.csv"
)


saved: submission.csv + submission_honest_stack.csv


## 10. Результаты, выводы и перспективы

Финальное решение заняло **216 место из 3520 — примерно Top 6% leaderboard**.

Проект демонстрирует не только использование готового алгоритма классификации, а полный workflow создания ensemble:

**Основной практический вывод проекта:** качество ансамбля определяется не количеством моделей само по себе, а сочетанием трёх факторов — силой отдельных моделей, разнообразием их ошибок и корректностью валидационного pipeline.

### Перспективы
Такое разделение также определяет дальнейшее развитие проекта: следующим естественным шагом было бы самостоятельно обучить несколько разнообразных базовых моделей на исходных признаках и сравнить собственную OOF-библиотеку со stack'ом предложенным другими участниками соревнования.